## BB84 Quantum Key Distribution Protocol

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

## Key Generaation

In [2]:
n = 20  # number of qubits

# Alice's random bits
alice_bits = np.random.randint(2, size=n)

# Alice's random bases (0 = Z, 1 = X)
alice_bases = np.random.randint(2, size=n)

## Encode qubits

In [3]:
def encode_qubits(bits, bases):
    circuits = []

    for i in range(len(bits)):
        qc = QuantumCircuit(1,1)

        # Encode bit
        if bits[i] == 1:
            qc.x(0)

        # Change basis
        if bases[i] == 1:
            qc.h(0)

        circuits.append(qc)

    return circuits

alice_circuits = encode_qubits(alice_bits, alice_bases)

In [4]:
def eve_intercept(circuits):
    eve_bases = np.random.randint(2, size=len(circuits))
    new_circuits = []

    for i, qc in enumerate(circuits):
        qc_eve = qc.copy()

        # Eve chooses random basis
        if eve_bases[i] == 1:
            qc_eve.h(0)

        qc_eve.measure(0,0)

        # Reset and resend (simplified)
        qc_new = QuantumCircuit(1,1)

        new_circuits.append(qc_new)

    return new_circuits, eve_bases

In [6]:
def bob_measure(circuits):
    bob_bases = np.random.randint(2, size=len(circuits))
    results = []

    sim = AerSimulator()

    for i, qc in enumerate(circuits):
        qc_bob = qc.copy()

        if bob_bases[i] == 1:
            qc_bob.h(0)

        qc_bob.measure(0,0)

        result = sim.run(qc_bob, shots=1).result()
        counts = result.get_counts()

        measured_bit = int(list(counts.keys())[0])
        results.append(measured_bit)

    return results, bob_bases

bob_results, bob_bases = bob_measure(alice_circuits)

In [7]:
shared_key = []

for i in range(n):
    if alice_bases[i] == bob_bases[i]:
        shared_key.append(alice_bits[i])

print("Shared Key:", shared_key)

Shared Key: [np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1)]


In [8]:
# Compare subset
sample_size = len(shared_key) // 2

alice_sample = shared_key[:sample_size]
bob_sample = shared_key[:sample_size]

error_rate = sum([a != b for a, b in zip(alice_sample, bob_sample)]) / sample_size

print("Error Rate:", error_rate)

Error Rate: 0.0
